In [1]:
import json
import requests

# API key used for the API access
api_key = "64686b9a75msha576b74f8ff0b56p1a63ebjsnf20f69b4fac9"

# List of outlet IDs
outlets = {
    "IGN": 56,
    "PC Gamer": 162,
    "GamesRadar+": 91,
    "Metro GameCentral": 75,
    "Eurogamer": 114,
    "Easy Allies": 394,
    "Game Informer": 35,
    "Forbes": 290,
    "GameSpot": 32,
    "Game Rant": 60
}

# Function to load platforms data from platforms.json
def load_platforms_data(filename="data/platforms.json"):
    with open(filename, 'r') as f:
        platforms_data = json.load(f)
    platforms_map = {platform['id']: platform['name'] for platform in platforms_data}
    return platforms_map

# Function to process reviews and save to JSON
def process_and_save_reviews(reviews, outlets, platforms_map, path="data/opencritic_reviews.json"):
    result = {}

    # Dictionary to store platform scores and counts
    platform_scores = {}
    platform_counts = {}

    for game_name, review in reviews:
        outlet_id = review['Outlet']['id']
        outlet_name = review['Outlet']['name']
        score = review.get('score')
        created_at = review['createdAt'][:10]  # Extract the date part (YYYY-MM-DD)
        authors = review.get('Authors', [])
        author_name = authors[0]['name'] if authors else None

        # Extract platforms from the review
        platforms = review.get('Platforms', [])
        if platforms:
            for platform in platforms:
                platform_id = platform['id']
                if platform_id in platforms_map:
                    platform_name = platforms_map[platform_id]
                    if platform_name not in platform_scores:
                        platform_scores[platform_name] = []
                        platform_counts[platform_name] = 0
                    if isinstance(score, (int, float)):
                        platform_scores[platform_name].append(score)
                        platform_counts[platform_name] += 1

        if game_name not in result:
            result[game_name] = {"outlets": {}, "platforms": {}, "average_score": None}

        if outlet_id in outlets.values():
            # Check if the score is a valid number
            if isinstance(score, (int, float)) and score is not None:
                result[game_name]["outlets"][outlet_name] = {
                    "score": score,
                    "review_url": review['externalUrl'],
                    "created_at": created_at,
                    "author": author_name  # Include author's name if available
                }

    # Calculate average scores for each platform
    for platform_name, scores in platform_scores.items():
        if scores:
            avg_score = sum(scores) / len(scores)
            result[game_name]["platforms"][platform_name] = {
                "average_score": avg_score,
                "count": platform_counts[platform_name]
            }

    # Calculate average score for the game
    all_scores = [outlet_data["score"] for outlet_data in result[game_name]["outlets"].values() if isinstance(outlet_data["score"], (int, float))]
    if all_scores:
        result[game_name]["average_score"] = sum(all_scores) / len(all_scores)

    # Save to JSON
    with open(path, 'w') as json_file:
        json.dump(result, json_file, indent=4)
    print(f"Reviews data saved to {path}")

# Load platforms data
platforms_map = load_platforms_data("data/platforms.json")

# Load all_reviews from the JSON file
with open('data/all_reviews.json', 'r') as json_file:
    all_reviews = json.load(json_file)

# Process reviews and save to JSON
process_and_save_reviews(all_reviews, outlets, platforms_map)

print("Processed reviews data saved to data/opencritic_reviews.json")


Reviews data saved to data/opencritic_reviews.json
Processed reviews data saved to data/opencritic_reviews.json
